In [44]:
# Cell 2: Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("OPENAI_API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")


OpenAI API key loaded and environment variable set successfully.


In [45]:
query = 'query = "How does the Oracle RAG approach differ from traditional vector stores regarding data security?"'
db_records = db_records =  [
    "Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines neural language models with secure retrieval systems.",
    "The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.",
    "This methodology ensures that sensitive corporate data never leaves the secure governance boundary of the Oracle environment.",
    "At the core of this architecture is the Oracle Database 23ai, which integrates vector search natively alongside relational data.",
    "Unlike external platforms that require data replication (ETL), Oracle allows for 'Integrated Vectorization', meaning vectors and business data coexist in real-time.",
    "The retrieval system utilizes Oracle AI Vector Search to fetch contextually relevant documents without the latency of external API calls.",
    "When a query is received, the Oracle system performs a similarity search directly on the resident data, ensuring zero data leakage.",
    "This component merges the retrieval outputs with the generative model, ensuring the final response is grounded in the organization's single source of truth.",
    "By keeping data in Oracle, the system eliminates the complexity of synchronizing a separate vector database with the operational database.",
    "The integrator seamlessly incorporates the retrieved Oracle data into the final text output, enhancing accuracy and compliance.",
    "This approach is particularly critical for industries like finance, healthcare, and government where data sovereignty is non-negotiable.",
    "Oracle AI Vector Search supports various distance metrics and indexing methods tailored for high-performance corporate workloads.",
    "The Multi-Agent System (MAS) can now orchestrate complex tasks by querying the Oracle database directly for both structured (SQL) and unstructured (Vector) data.",
    "This response is influenced by real-time corporate data, making it far more accurate than models relying on stale, externally indexed data.",
    "Retrieval Augmented Generation (RAG) in Oracle adapts dynamically; as soon as a record is updated in the database, it is immediately available for retrieval.",
    "This eliminates the 'stale data' problem common in architectures that rely on external vector stores like Pinecone or FAISS.",
    "With access to the vast, secure repository of Oracle data, the RAG system provides nuanced answers grounded in the company's actual operational reality.",
    "The challenge of maintaining a separate, high-quality database of retrievable texts is solved by converging vectors and data into one Oracle platform.",
    "Furthermore, Oracle's robust security features (Row-Level Security, auditing) automatically apply to the vector retrieval process.",
    "In summary, Oracle-based RAG represents the maturity of AI: merging the best of generative technologies with the security and reliability of enterprise data systems.",
    "An Oracle Vector Store is not a separate silo; it is a capability of the converged database that treats vectors as a native data type."
]


In [46]:
import openai
from openai import OpenAI

client = OpenAI()
gptmodel="gpt-5.2"

def call_llm_with_full_text(itext):
    # Join all lines to form a single string
    text_input = '\n'.join(itext)
    prompt = f"Please elaborate on the following content:\n{text_input}"

    try:
      response = client.chat.completions.create(
         model=gptmodel,
         messages=[
            {"role": "system", "content": "You are an expert Natural Language Processing exercise expert."},
            {"role": "assistant", "content": "1.You can explain read the input and answer in detail"},
            {"role": "user", "content": prompt}
         ],
         temperature=0.1  # Add the temperature parameter here and other parameters you need
        )
      return response.choices[0].message.content.strip()
    except Exception as e:
        return str(e)

In [68]:
import textwrap
def print_formatted_response(response):
    wrapper = textwrap.TextWrapper(width=80) #set 80 colummn wide
    wrapper_text = wrapper.fill(text = response)
    # Print the formatted response with a header and footer
    print("Response:")
    print(wrapped_text)
  #db_record
paragraph = ' '.join(db_records)
wrapped_text = textwrap.fill(paragraph, width=80)
wrapped_text

"Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines\nneural language models with secure retrieval systems. The Oracle strategy for\nRAG represents a paradigm shift: instead of moving data to external vector\nstores, it brings AI vector search directly to the database. This methodology\nensures that sensitive corporate data never leaves the secure governance\nboundary of the Oracle environment. At the core of this architecture is the\nOracle Database 23ai, which integrates vector search natively alongside\nrelational data. Unlike external platforms that require data replication (ETL),\nOracle allows for 'Integrated Vectorization', meaning vectors and business data\ncoexist in real-time. The retrieval system utilizes Oracle AI Vector Search to\nfetch contextually relevant documents without the latency of external API calls.\nWhen a query is received, the Oracle system performs a similarity search\ndirectly on the resident data, ensuring zero data leakage. This com

In [48]:
import spacy
import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet
from collections import Counter
import numpy as np

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name())
    return synonyms

def preprocess_text(text):
    doc = nlp(text.lower())
    lemmatized_words = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        lemmatized_words.append(token.lemma_)
    return lemmatized_words

def expand_with_synonyms(words):
    expanded_words = words.copy()
    for word in words:
        expanded_words.extend(get_synonyms(word))
    return expanded_words

def calculate_enhanced_similarity(text1, text2):
    # Preprocess and tokenize texts
    words1 = preprocess_text(text1)
    words2 = preprocess_text(text2)

    # Expand with synonyms
    words1_expanded = expand_with_synonyms(words1)
    words2_expanded = expand_with_synonyms(words2)

    # Count word frequencies
    freq1 = Counter(words1_expanded)
    freq2 = Counter(words2_expanded)

    # Create a set of all unique words
    unique_words = set(freq1.keys()).union(set(freq2.keys()))

    # Create frequency vectors
    vector1 = [freq1[word] for word in unique_words]
    vector2 = [freq2[word] for word in unique_words]

    # Convert lists to numpy arrays
    vector1 = np.array(vector1)
    vector2 = np.array(vector2)

    # Calculate cosine similarity
    cosine_similarity = np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))

    return cosine_similarity

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [49]:
def find_best_match_keyword_search(query, db_records):
    best_score = 0
    best_record = None

    # Split the query into individual keywords
    query_keywords = set(query.lower().split())

    # Iterate through each record in db_records
    for record in db_records:
        # Split the record into keywords
        record_keywords = set(record.lower().split())

        # Calculate the number of common keywords
        common_keywords = query_keywords.intersection(record_keywords)
        current_score = len(common_keywords)

        # Update the best score and record if the current score is higher
        if current_score > best_score:
            best_score = current_score
            best_record = record

    return best_score, best_record

# Assuming 'query' and 'db_records' are defined in previous cells in your Colab notebook
best_keyword_score, best_matching_record = find_best_match_keyword_search(query, db_records)

print(f"Best Keyword Score: {best_keyword_score}")
print_formatted_response(best_matching_record)

Best Keyword Score: 5
Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero dat

In [50]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    # Use the default settings to match the book's explanation
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([text1, text2])
    similarity = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return similarity[0][0]

In [51]:

# Cosine Similarity
score = calculate_cosine_similarity(query, best_matching_record)
print(f"Best Cosine Similarity Score: {score:.3f}")

Best Cosine Similarity Score: 0.233


In [52]:
# Enhanced Similarity
response = best_matching_record
print(query,": ", response)
similarity_score = calculate_enhanced_similarity(query, response)
print(f"Enhanced Similarity:, {similarity_score:.3f}")

query = "How does the Oracle RAG approach differ from traditional vector stores regarding data security?" :  The Oracle strategy for RAG represents a paradigm shift: instead of moving data to external vector stores, it brings AI vector search directly to the database.
Enhanced Similarity:, 0.488


In [53]:

augmented_input=query+ ": "+ best_matching_record

print_formatted_response(augmented_input)

Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This compon

In [54]:
# Call the function and print the result
llm_response = call_llm_with_full_text(augmented_input)
print_formatted_response(llm_response)

Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This compon

In [55]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def setup_vectorizer(records):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(records)

    # Convert the TF-IDF matrix to a DataFrame for display purposes
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

    # Display the DataFrame
    print(tfidf_df)

    return vectorizer, tfidf_matrix

vectorizer, tfidf_matrix = setup_vectorizer(db_records)


        23ai    access  accuracy  accurate    actual   adapts     agent  \
0   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
1   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
2   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
3   0.290558  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
4   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
5   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
6   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
7   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
8   0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
9   0.000000  0.000000  0.283723  0.000000  0.000000  0.00000  0.000000   
10  0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
11  0.000000  0.000000  0.000000  0.000000  0.000000  0.00000  0.000000   
12  0.000000  0.000000  0

In [56]:
augmented_input=query+": "+best_matching_record

print_formatted_response(augmented_input)

Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This compon

In [57]:
#day 3 where   modular rag
# for example :
#   keywords search , vector search , indexed search :::


In [58]:
#keyword search

docs = ["apple is a fruit", "iphone is a phone"]
query = "apple"
results = [doc for doc in docs if query in doc]
print(results)



['apple is a fruit']


In [59]:
# indexes search [tf- idf ]
#exmaple :
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = ["apple is a fruit", "iphone is a phone"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)
# print(tfidf_matrix)

query = "apple"
query_vector = vectorizer.transform([query])

cosine_similarities = cosine_similarity(query_vector, tfidf_matrix)
print(cosine_similarities)




[[0.6316672 0.       ]]


In [60]:
# Vector Search (Embeddings)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


#USER  QUERY FIND apple
docs = ["apple is a fruit", "iphone is a phone"]
doc_vectors = model.encode(docs)
# print("doc_vectors",doc_vectors)

query = "best fruit"
query_vector = model.encode([query])
# print("query_vector",query_vector)

similarity = cosine_similarity(query_vector, doc_vectors)
print("similarity", similarity)






similarity [[0.64727676 0.15154177]]


In [66]:
# pipelines of RAG SEARCHES keyword , vector ,  indexed  :

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class RetrievalComponenets():

  def __init__(self, method = "vector" ):
    self.method = method
    if self.method == "vector" or self.method == "indexed":
      self.vectorizer = TfidfVectorizer()
      self.tfidf_matrix = None

  def fit(self,records):
   self.documents = records  # Initialize self.documents here
   if self.method == 'vector' or self.method == 'indexed':
    self.tfidf_matrix = self.vectorizer.fit_transform(records)

  def retrieve(self, query):
        if self.method == 'keyword':
            return self.keyword_search(query)
        elif self.method == 'vector':
            return self.vector_search(query)
        elif self.method == 'indexed':
            return self.indexed_search(query)

  # keyword search :

  def keyword_search(self, query):
      best_score = 0
      best_record = None
      query_keywords = set(query.lower().split())
      for index, doc in enumerate(self.documents):
          doc_keywords = set(doc.lower().split())
          common_keywords = query_keywords.intersection(doc_keywords)
          score = len(common_keywords)
          if score > best_score:
              best_score = score
              best_record = self.documents[index]
      return best_record

  # vector search :

  def vector_search(self, query):
    query_tfidf = self.vectorizer.transform([query])
    similarities = cosine_similarity(query_tfidf, self.tfidf_matrix)
    best_index = similarities.argmax()
    return self.documents[best_index]

  def indexed_search(self, query):
          # Assuming the tfidf_matrix is precomputed and stored
          query_tfidf = self.vectorizer.transform([query])
          similarities = cosine_similarity(query_tfidf, self.tfidf_matrix)
          best_index = similarities.argmax()
          return self.documents[best_index]



retrivel = RetrievalComponenets(method="indexed") # keyword , vector ,  indexed
retrivel.fit(db_records)

best_matching_record = retrivel.retrieve(query)

print_formatted_response(best_matching_record)






Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This compon

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def setup_vectorizer(records):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(records)
    return vectorizer, tfidf_matrix

def find_best_match_indexed(query, vectorizer, tfidf_matrix):
    query_tfidf = vectorizer.transform([query])
    similarities = cosine_similarity(query_tfidf, tfidf_matrix)
    best_index = similarities.argmax()  # Get the index of the highest similarity score
    best_score = similarities[0, best_index]
    return best_score, best_index

vectorizer, tfidf_matrix = setup_vectorizer(db_records)

best_similarity_score, best_index = find_best_match_indexed(query, vectorizer, tfidf_matrix)
best_matching_record = db_records[best_index]

print_formatted_response(best_matching_record)

Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This compon

In [73]:

# Cosine Similarity
print(f"Best Cosine Similarity Score: {best_similarity_score:.3f}")
print_formatted_response(best_matching_record)

Best Cosine Similarity Score: 0.243
Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ens

In [75]:
#enhance tht similairty
response = best_matching_record
print(query,": ", response)
similarity_score = calculate_enhanced_similarity(query, response)
print(f"Enhanced Similarity:, {similarity_score:.3f}")

best fruit :  In summary, Oracle-based RAG represents the maturity of AI: merging the best of generative technologies with the security and reliability of enterprise data systems.
Enhanced Similarity:, 0.363


In [78]:
#argument inputs
augmented_input= query+ " "+ best_matching_record

In [79]:
#generater :
# Call the function and print the result
llm_response = call_llm_with_full_text(augmented_input)
print_formatted_response(llm_response)

Response:
Retrieval Augmented Generation (RAG) is a hybrid AI approach that combines
neural language models with secure retrieval systems. The Oracle strategy for
RAG represents a paradigm shift: instead of moving data to external vector
stores, it brings AI vector search directly to the database. This methodology
ensures that sensitive corporate data never leaves the secure governance
boundary of the Oracle environment. At the core of this architecture is the
Oracle Database 23ai, which integrates vector search natively alongside
relational data. Unlike external platforms that require data replication (ETL),
Oracle allows for 'Integrated Vectorization', meaning vectors and business data
coexist in real-time. The retrieval system utilizes Oracle AI Vector Search to
fetch contextually relevant documents without the latency of external API calls.
When a query is received, the Oracle system performs a similarity search
directly on the resident data, ensuring zero data leakage. This compon

In [80]:
# intergrate the db with google collab :
from google.colab import drive
drive.mount('/content/drive')
if unzip_wallet==True:
  !unzip -o "/content/drive/MyDrive/files/oracle/Wallet_*.zip" -d "/content/drive/MyDrive/files/oracle"
wallet_path = "/content/drive/MyDrive/oracle_wallet"

MessageError: Error: credential propagation was unsuccessful